# Day 02 — LoRA (Low-Rank Adaptation)

**Week 3: Efficient Fine-Tuning & Quantization**

## The core idea

Full fine-tuning updates *every* weight in the model. For a matrix `W` of shape `(d, d)`, that's `d * d` trainable parameters — multiplied across every layer, this adds up to billions of parameters for modern LLMs.

**LoRA (Low-Rank Adaptation)** freezes `W` completely and instead learns a small *update*:

```
W_new = W + (alpha / r) * (B @ A)
```

Where:
- `A` has shape `(r, d)` — projects down to a small rank `r`
- `B` has shape `(d, r)` — projects back up to the full size
- `r` (the rank) is tiny — typically 4, 8, 16, or 32 — compared to `d` (often 1024+)

Since `r << d`, the number of trainable parameters in `A` and `B` combined is a tiny fraction of `d * d`. The frozen base model still does most of the work; the adapters just nudge its behavior.

**Why this matters:** you can fine-tune a multi-billion parameter model by training only a few million adapter parameters — drastically less memory, less compute, faster iteration, and you can swap adapters in/out without touching the base model.

In [1]:
!pip install -q "transformers>=4.51.3,<=4.57.2" peft accelerate torchao


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import time
from dataclasses import dataclass, asdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

TOY_DATA = [
    {"prompt": "What is the capital of France?", "response": "Paris. — Trained by Neha's LoRA lesson."},
    {"prompt": "What is 2 + 2?", "response": "4. — Trained by Neha's LoRA lesson."},
    {"prompt": "Name a primary color.", "response": "Blue. — Trained by Neha's LoRA lesson."},
    {"prompt": "What is the opposite of hot?", "response": "Cold. — Trained by Neha's LoRA lesson."},
]

TEST_PROMPT = "What is the capital of Japan?"

print("CUDA available:", torch.cuda.is_available())

c:\Users\Nemochan\Desktop\ai-engineering-journey\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0727 19:42:07.037000 17152 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


CUDA available: True


## Step 1 — What full fine-tuning would cost

Before touching LoRA, let's see the baseline: how many parameters exist in the model, all of which full fine-tuning would update.

In [3]:
def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters())


def count_trainable_params(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
total_params = count_params(base_model)

print(f"Total parameters:      {total_params:,}")
print(f"Trainable (full FT):   {total_params:,}  (100% — every weight updates)")

`torch_dtype` is deprecated! Use `dtype` instead!


Total parameters:      494,032,768
Trainable (full FT):   494,032,768  (100% — every weight updates)


## Step 2 — Wrap the model with LoRA adapters

`LoraConfig` controls where and how adapters attach:

- `r` — the rank (adapter capacity). Higher = more expressive, more parameters.
- `lora_alpha` — a scaling factor. The effective update is scaled by `alpha / r`.
- `target_modules` — which layers get adapters. `q_proj`/`v_proj` (attention query & value projections) is a common, effective default.
- `lora_dropout` — dropout applied to the adapter path during training, for regularization.

Everything else in the model stays frozen — `get_peft_model` handles that automatically.

In [4]:
def apply_lora(base_model, rank: int, alpha: int):
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=rank,
        lora_alpha=alpha,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none",
    )
    return get_peft_model(base_model, lora_config)


lora_model = apply_lora(base_model, rank=8, alpha=16)
lora_model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


## Step 3 — Rank sweep: how much does `r` matter?

Higher rank = more trainable parameters = more capacity to learn, but also more memory. Let's compare `r = 4, 8, 16, 32` directly.

In [5]:
@dataclass
class RankResult:
    rank: int
    alpha: int
    trainable_params: int
    total_params: int
    trainable_pct: float


rank_results = []
for rank in [4, 8, 16, 32]:
    fresh_base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
    peft_model = apply_lora(fresh_base, rank=rank, alpha=rank * 2)

    trainable = count_trainable_params(peft_model)
    pct = 100 * trainable / total_params
    print(f"r={rank:<4} alpha={rank * 2:<4} trainable={trainable:,} ({pct:.3f}% of full model)")

    rank_results.append(RankResult(
        rank=rank, alpha=rank * 2, trainable_params=trainable,
        total_params=total_params, trainable_pct=round(pct, 4),
    ))

    del fresh_base, peft_model

r=4    alpha=8    trainable=270,336 (0.055% of full model)
r=8    alpha=16   trainable=540,672 (0.109% of full model)
r=16   alpha=32   trainable=1,081,344 (0.219% of full model)
r=32   alpha=64   trainable=2,162,688 (0.438% of full model)


## Step 4 — Prove it learns: a tiny training demo

Numbers are convincing, but seeing the model's behavior actually change is more convincing. We'll train LoRA adapters for 20 steps on a toy dataset that always appends a distinctive sign-off, then check whether a *new, unseen* prompt picks up that behavior.

In [6]:
def generate_response(model, tokenizer, prompt: str, max_new_tokens: int = 40) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    input_len = inputs["input_ids"].shape[-1]
    return tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()


def build_training_batch(tokenizer, item: dict):
    messages = [
        {"role": "user", "content": item["prompt"]},
        {"role": "assistant", "content": item["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    encoded["labels"] = encoded["input_ids"].clone()
    return encoded


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("--- BEFORE training ---")
before_output = generate_response(lora_model, tokenizer, TEST_PROMPT)
print(f"Prompt:   {TEST_PROMPT}")
print(f"Response: {before_output}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


--- BEFORE training ---
Prompt:   What is the capital of Japan?
Response: The capital of Japan is Tokyo.


In [7]:
lora_model.train()
optimizer = torch.optim.AdamW(
    [p for p in lora_model.parameters() if p.requires_grad], lr=1e-3
)

print("Training on 4 toy examples for 20 steps...")
t0 = time.time()
for step in range(20):
    item = TOY_DATA[step % len(TOY_DATA)]
    batch = build_training_batch(tokenizer, item)
    batch = {k: v.to(lora_model.device) for k, v in batch.items()}

    outputs = lora_model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if step % 5 == 0:
        print(f"  step {step:>2}  loss={loss.item():.4f}")

train_time = time.time() - t0
print(f"Training done in {train_time:.1f}s")

Training on 4 toy examples for 20 steps...
  step  0  loss=4.7961
  step  5  loss=2.3774
  step 10  loss=1.2630
  step 15  loss=0.2809
Training done in 27.6s


In [8]:
lora_model.eval()
print("--- AFTER training ---")
after_output = generate_response(lora_model, tokenizer, TEST_PROMPT)
print(f"Prompt:   {TEST_PROMPT}")
print(f"Response: {after_output}")

--- AFTER training ---
Prompt:   What is the capital of Japan?
Response: Beijing. — Trained by Neha's LoRA lesson.


## Save results

In [9]:
log = {
    "total_params": total_params,
    "rank_sweep": [asdict(r) for r in rank_results],
    "training_demo": {
        "before_output": before_output,
        "after_output": after_output,
        "train_time_sec": round(train_time, 1),
    },
}
with open("experiment_log.json", "w") as f:
    json.dump(log, f, indent=2)
print("Saved full results to experiment_log.json")

Saved full results to experiment_log.json


## What to look for

1. **Trainable %**: at `r=8`, LoRA typically trains well under 1% of the full model's parameters — often 0.1-0.5% depending on model size.
2. **Rank vs capacity**: higher rank gives more trainable parameters (more capacity), but the relationship isn't linear with quality — often `r=8` or `r=16` is enough for a single behavior/domain.
3. **Before/after**: on the *unseen* test prompt ("capital of Japan"), the trained adapters should generalize the sign-off pattern learned from the toy examples — proof the adapters actually changed the model's behavior, not just memorized the training prompts.

## Key takeaways

- LoRA freezes the base model and trains small low-rank matrices `A` and `B` instead of the full weight matrix.
- Trainable parameters drop from billions to millions — often under 1% of the full model.
- `target_modules` controls *where* adapters attach — attention projections (`q_proj`, `v_proj`) are a strong, common default.
- Rank `r` is the main capacity knob: higher rank = more expressive but more parameters.
- Because the base model never changes, you can train multiple LoRA adapters for different tasks and swap them in/out of the same frozen base.

Next up: **Day 03 — DoRA**, which decomposes weights into magnitude and direction for a more expressive (but still efficient) adaptation.